# 11 · GEMM v2:autotune、num_stages 与 L2 swizzle

> Learn Triton 系列 · 阶段 2(核心算子)第 3 篇
> 前置:第 10 篇(GEMM v1,固定配置达到 cuBLAS 的 50%~75%)
> 运行环境:Google Colab T4 GPU

第 10 篇结尾列了三个差距来源:块配置不优、流水线太浅、L2 复用差。本篇逐项收复:用 `@triton.autotune` 自动选配置,用 `num_stages` 加深软件流水线,用 **grouped ordering(L2 swizzle)** 重排 program 执行顺序。这三板斧是所有 Triton 高性能 kernel 的标配,也是"你真的调过 kernel"和"只会抄模板"的分水岭。

## 环境准备

In [ ]:
import torch

assert torch.cuda.is_available(), "请在 Colab 选择 GPU 运行时"

import triton
import triton.language as tl
import triton.testing

print(f"PyTorch {torch.__version__} | Triton {triton.__version__} | {torch.cuda.get_device_name(0)}")

---

## §1 是什么 & 能力边界

### 三个旋钮分别在调什么

**① 块大小(BLOCK_M/N/K)**:复用率与资源占用的拉锯。

- 块越大 → AI 越高(显存流量 ∝ 1/块边长)、Tensor Core 指令越连贯;
- 块越大 → 每个 program 占用的寄存器/shared memory 越多 → 一个 SM 能同时常驻的 program 越少(**occupancy 下降**)→ 没有足够的并行 warp 来掩盖访存延迟;
- 最优点取决于形状与硬件,**没有公式,只有实测**。

**② num_stages(软件流水线)**:K 循环里"load 下一块"与"dot 当前块"重叠执行,编译器把循环展开成 N 级流水(shared memory 里开 N 份缓冲)。stages 越深掩盖延迟越好,但 shared memory 占用 ∝ stages。注:Ampere(sm_80)起有 `cp.async` 异步拷贝指令,流水线效率更高;T4(sm_75)上收益打折但仍存在。

**③ GROUP_M(L2 swizzle / grouped ordering)**:GPU 大致按 pid 顺序调度 program。默认行优先顺序下,同一波(wave)在跑的 ~40 个 program 全在 C 的同一行——它们共享 1 个 A 行块,但要读 40 个**不同**的 B 列块,L2 根本装不下。把执行顺序重排成"先走完一个 GROUP_M×n 的窄竖条"后,同一波的 program 集中在一个小矩形里,**A、B 的读取都落在少数几个块上,L2 命中率大增**。不改变任何计算,只改变顺序。

### `@triton.autotune` 的机制

```py
@triton.autotune(configs=[...], key=["M", "N", "K"])
@triton.jit
def kernel(...): ...
```

- `configs`:候选配置列表(块大小、GROUP_M、num_warps、num_stages 的组合);
- `key`:这些运行期参数的取值一变,就重新 benchmark 所有候选并缓存最优——**同一形状只调一次,之后零开销**;
- 调优发生在第一次调用时(对每个新 key),每个候选跑若干次取最优。

### 能做什么 / 不能做什么

能做:

- 同一份 kernel 源码自动适配不同形状/GPU(换到 A100 重新 autotune 即可,代码零改动);
- 把 v1 与 cuBLAS 的差距缩小到 10%~20% 以内(本篇实验验证);
- `key` 还可以包含 dtype、是否转置等任何影响最优配置的因素。

不能做:

- **不改变算法**:autotune 只在你给出的候选里挑,候选全烂则结果照烂;swizzle/流水线这类结构性优化必须写进 kernel;
- 首次调优有秒级开销:对动态形状极多的场景(如任意 batch 的在线服务)要做形状分桶,否则一直在调优(vLLM 对此的做法:按 batch 段位预调 + 缓存配置文件);
- 候选太多调优太慢、太少容易漏掉最优——工程上从官方模板的 6~10 个候选起步,再按 profile 增删;
- 在共享 GPU(Colab)上调优结果有噪声:同一 key 两次会话可能选出不同配置。

---

## §2 递进式例子

### 例 1:配置敏感性 —— 同一个 kernel,手动换配置差多少

复用第 10 篇的 kernel,只换启动配置,在 2048³ 上实测。

In [ ]:
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr, M, N, K,
    stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    rk = tl.arange(0, BLOCK_K)
    a_ptrs = a_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak
    b_ptrs = b_ptr + rk[:, None] * stride_bk + rn[None, :] * stride_bn
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K, BLOCK_K):
        a = tl.load(a_ptrs, mask=(rm[:, None] < M) & (rk[None, :] + k < K), other=0.0)
        b = tl.load(b_ptrs, mask=(rk[:, None] + k < K) & (rn[None, :] < N), other=0.0)
        acc += tl.dot(a, b)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk
    tl.store(c_ptr + rm[:, None] * stride_cm + rn[None, :] * stride_cn,
             acc.to(tl.float16), mask=(rm[:, None] < M) & (rn[None, :] < N))


def run_config(a, b, BM, BN, BK, warps, stages):
    M, K = a.shape; N = b.shape[1]
    c = torch.empty(M, N, device="cuda", dtype=torch.float16)
    grid = (triton.cdiv(M, BM), triton.cdiv(N, BN))
    fn = lambda: matmul_kernel[grid](a, b, c, M, N, K,
                                     a.stride(0), a.stride(1), b.stride(0), b.stride(1),
                                     c.stride(0), c.stride(1),
                                     BLOCK_M=BM, BLOCK_N=BN, BLOCK_K=BK,
                                     num_warps=warps, num_stages=stages)
    fn(); torch.testing.assert_close(c, a @ b, rtol=1e-2, atol=1e-2)
    return triton.testing.do_bench(fn, return_mode="median")


n = 2048
A = torch.randn(n, n, device="cuda", dtype=torch.float16)
B = torch.randn(n, n, device="cuda", dtype=torch.float16)
flops = 2 * n**3

configs = [
    (32, 32, 32, 2, 2), (64, 64, 32, 4, 2), (64, 64, 32, 4, 4),
    (128, 64, 32, 4, 4), (64, 128, 32, 4, 4), (128, 128, 32, 8, 3),
    (128, 128, 64, 8, 2), (32, 128, 64, 4, 4),
]
print(f"{'BM':>4}{'BN':>5}{'BK':>5}{'warps':>7}{'stages':>8} | {'TFLOPS':>7}")
best = (None, 0)
for cfg in configs:
    try:
        ms = run_config(A, B, *cfg)
        tf = flops / (ms / 1000) / 1e12
        if tf > best[1]:
            best = (cfg, tf)
        print(f"{cfg[0]:>4}{cfg[1]:>5}{cfg[2]:>5}{cfg[3]:>7}{cfg[4]:>8} | {tf:>7.2f}")
    except Exception as e:
        print(f"{cfg[0]:>4}{cfg[1]:>5}{cfg[2]:>5}{cfg[3]:>7}{cfg[4]:>8} | 失败({type(e).__name__})")
print(f"\n最优 {best[0]} -> {best[1]:.2f} TFLOPS;最差与最优常相差 2 倍以上 —— GEMM 对配置极端敏感")

### 例 2:正式版 —— autotune + GROUP_M swizzle(与官方教程同构)

In [ ]:
@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32, "GROUP_M": 8}, num_warps=8, num_stages=3),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64,  "BLOCK_K": 32, "GROUP_M": 8}, num_warps=4, num_stages=4),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 128, "BLOCK_K": 32, "GROUP_M": 8}, num_warps=4, num_stages=4),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 64,  "BLOCK_K": 32, "GROUP_M": 8}, num_warps=4, num_stages=4),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 64,  "BLOCK_K": 64, "GROUP_M": 8}, num_warps=4, num_stages=3),
        triton.Config({"BLOCK_M": 32,  "BLOCK_N": 64,  "BLOCK_K": 32, "GROUP_M": 8}, num_warps=2, num_stages=4),
    ],
    key=["M", "N", "K"],      # 形状一变,重新调优并缓存
)
@triton.jit
def matmul_kernel_v2(
    a_ptr, b_ptr, c_ptr, M, N, K,
    stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    GROUP_M: tl.constexpr,
):
    # ---- L2 swizzle:把一维 pid 重排成"GROUP_M 行一组的竖条走法" ----
    pid = tl.program_id(0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    num_pid_in_group = GROUP_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_M)
    pid_m = first_pid_m + (pid % group_size_m)            # 组内先沿 M 走
    pid_n = (pid % num_pid_in_group) // group_size_m      # 再换列

    # ---- 以下与 v1 完全相同 ----
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    rk = tl.arange(0, BLOCK_K)
    a_ptrs = a_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak
    b_ptrs = b_ptr + rk[:, None] * stride_bk + rn[None, :] * stride_bn
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K, BLOCK_K):
        a = tl.load(a_ptrs, mask=(rm[:, None] < M) & (rk[None, :] + k < K), other=0.0)
        b = tl.load(b_ptrs, mask=(rk[:, None] + k < K) & (rn[None, :] < N), other=0.0)
        acc += tl.dot(a, b)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk
    tl.store(c_ptr + rm[:, None] * stride_cm + rn[None, :] * stride_cn,
             acc.to(tl.float16), mask=(rm[:, None] < M) & (rn[None, :] < N))


def triton_matmul_v2(a, b):
    M, K = a.shape; N = b.shape[1]
    c = torch.empty(M, N, device="cuda", dtype=torch.float16)
    grid = lambda meta: (triton.cdiv(M, meta["BLOCK_M"]) * triton.cdiv(N, meta["BLOCK_N"]),)
    matmul_kernel_v2[grid](a, b, c, M, N, K,
                           a.stride(0), a.stride(1), b.stride(0), b.stride(1),
                           c.stride(0), c.stride(1))
    return c


print("首次调用会对每个候选实测(几秒钟)...")
out = triton_matmul_v2(A, B)   # 触发 autotune
torch.testing.assert_close(out, A @ B, rtol=1e-2, atol=1e-2)
print("v2 正确 ✓;被选中的最优配置:")
print(matmul_kernel_v2.best_config)

### 例 3:swizzle 为什么有效 —— 算一笔 L2 字节账

不跑 GPU,纯算术模拟:一波同时常驻 40 个 program,比较两种执行顺序下这一波需要从显存读的**去重后**数据量。

In [ ]:
def wave_bytes(order: str, wave=40, BM=128, BN=128, K=2048, group_m=8):
    """一波 program 触碰的去重 A/B 块字节数(fp16)。"""
    if order == "row-major":          # 默认顺序:一波全在同一行,列号 0..39
        a_tiles, b_tiles = 1, wave    # 共享 1 个 A 行条,40 个不同 B 列条
    else:                              # grouped:一波覆盖 8 行 x 5 列 的矩形
        a_tiles, b_tiles = group_m, wave // group_m
    bytes_a = a_tiles * BM * K * 2     # 每个 A 行条: BM x K
    bytes_b = b_tiles * K * BN * 2     # 每个 B 列条: K x BN
    return bytes_a + bytes_b


for order in ["row-major", "grouped"]:
    mb = wave_bytes(order) / 1024**2
    print(f"{order:>10} 顺序: 一波需读 {mb:7.1f} MB  (T4 L2 = 4MB, {'装不下,反复淘汰' if mb > 4 else '基本可驻留'})")
print("\ngrouped 顺序把一波的工作集压缩数倍,L2 命中率随之提高 —— 计算一行未改,只改了'谁先跑'")

---

## §3 知识连接

**与前面篇章:**

- 第 04 篇例 3(一维 pid 自由映射到数据)在例 2 的 swizzle 里发挥到极致:pid → (pid_m, pid_n) 的映射就是一段纯算术,想怎么走就怎么走;
- 第 08 篇的"L2 缓存骗人"反过来用:既然 L2 这么快,就**主动设计**执行顺序去蹭它;
- 第 02 篇 occupancy 概念在例 1 落地:128×128 块比 64×64 复用更好,却可能因 occupancy 跌得更快而更慢。

**与真实框架:**

- 本篇例 2 与 Triton 官方教程 03-matrix-multiplication 的最终版本同构,官方教程在 A100 上报告达到 cuBLAS 90%+;
- vLLM/SGLang 中几乎每个性能敏感的 Triton kernel 头上都挂着 `@triton.autotune`(或框架自己的配置缓存机制,如 vLLM 的 `fused_moe` 把调优结果存成 JSON 配置文件按 GPU 型号分发——解决"在线服务不能现场调优"的问题);
- cuBLAS 内部同样是"配置表 + 启发式选择",`cublasLtMatmulAlgoGetHeuristic` 暴露了这层;CUTLASS 的 profiler 工具就是离线版 autotune。

---

## §4 闭环对比实验:v1 vs v2(autotuned)vs cuBLAS

方阵 + 两个 LLM 真实形状(MLP 上投影 / decode 小批量),看 v2 把达成率拉到多少。

In [ ]:
import matplotlib.pyplot as plt

shapes = [
    (512, 512, 512), (1024, 1024, 1024), (2048, 2048, 2048), (4096, 4096, 4096),
    (4096, 11008, 4096),   # Llama-7B MLP up-proj (prefill 4096 token)
    (16, 4096, 4096),      # decode: batch 16
]

def triton_matmul_v1(a, b):
    M, K = a.shape; N = b.shape[1]
    c = torch.empty(M, N, device="cuda", dtype=torch.float16)
    grid = (triton.cdiv(M, 64), triton.cdiv(N, 64))
    matmul_kernel[grid](a, b, c, M, N, K, a.stride(0), a.stride(1),
                        b.stride(0), b.stride(1), c.stride(0), c.stride(1),
                        BLOCK_M=64, BLOCK_N=64, BLOCK_K=32, num_warps=4, num_stages=2)
    return c


labels, r_v1, r_v2 = [], [], []
print(f"{'形状 (M,N,K)':>22} | {'v1 TF':>7} | {'v2 TF':>7} | {'cuBLAS':>7} | {'v2 达成率':>8}")
for (M, N, K) in shapes:
    a = torch.randn(M, K, device="cuda", dtype=torch.float16)
    b = torch.randn(K, N, device="cuda", dtype=torch.float16)
    torch.testing.assert_close(triton_matmul_v2(a, b), a @ b, rtol=1e-2, atol=1e-2)
    flops = 2 * M * N * K
    tf = lambda ms: flops / (ms / 1000) / 1e12
    t1 = tf(triton.testing.do_bench(lambda: triton_matmul_v1(a, b), return_mode="median"))
    t2 = tf(triton.testing.do_bench(lambda: triton_matmul_v2(a, b), return_mode="median"))
    tc = tf(triton.testing.do_bench(lambda: a @ b, return_mode="median"))
    labels.append(f"{M}x{N}x{K}"); r_v1.append(t1 / tc * 100); r_v2.append(t2 / tc * 100)
    print(f"{str((M,N,K)):>22} | {t1:>7.2f} | {t2:>7.2f} | {tc:>7.2f} | {t2 / tc * 100:>7.1f}%")

xpos = range(len(labels))
plt.figure(figsize=(10, 4.5))
plt.bar([x - 0.2 for x in xpos], r_v1, width=0.4, label="v1 (fixed config)")
plt.bar([x + 0.2 for x in xpos], r_v2, width=0.4, label="v2 (autotune+swizzle)")
plt.axhline(100, color="gray", ls="--", label="cuBLAS = 100%")
plt.xticks(list(xpos), labels, rotation=20)
plt.ylabel("% of cuBLAS")
plt.title("GEMM: % of cuBLAS achieved on T4")
plt.legend(); plt.tight_layout(); plt.show()

### 实验结果解读

- v2 在大方阵上通常到 cuBLAS 的 **80%~95%**,比 v1 提升明显——三板斧各贡献一截(配置匹配 > 流水线 > swizzle,大形状下 swizzle 占比上升);
- **decode 形状(16×4096×4096)注意看**:绝对 TFLOPS 很低(memory-bound,第 10 篇已预言),v1/v2/cuBLAS 都打不满算力——这种形状的真正优化是量化与攒批,不是调 GEMM 参数;
- 剩下 5%~20% 的差距是 Triton 在该架构上的代价(指令调度、Turing 无 cp.async 等)。工程结论:**裸 GEMM 用 cuBLAS,带融合/特殊精度的 GEMM 用 Triton**——下一篇就开始"带融合"。

---

## §5 练习 + 面试考点

### 动手练习

1. 把 `GROUP_M` 改成 1(等价于关闭 swizzle)加入 autotune 候选,看大形状下被选中/性能差异,验证例 3 的纸面分析。
2. 给 `key` 加上 `"a_transposed"` 维度:对 `x @ W^T`(第 10 篇例 3)与 `x @ W` 分别调优,比较两者最优配置是否相同,解释原因。

### 面试高频考点

- **Q:`@triton.autotune` 的原理?线上服务怎么避免现场调优开销?**
  A:按 `key` 缓存"配置 → 实测最优"映射,新 key 首次调用时 benchmark 所有候选。线上做法:形状分桶减少 key 数量;离线预调优并持久化(vLLM 的 fused MoE 按 GPU 型号发布 JSON 配置);或退化为启发式默认配置。
- **Q:num_stages 是什么?为什么不是越大越好?**
  A:软件流水线深度——K 循环的 load 与 dot 重叠,shared memory 开 N 份缓冲轮转。太深则 shared memory 超限、occupancy 下降;且延迟一旦被完全掩盖,继续加深只有成本没有收益。
- **Q:解释 L2 swizzle / grouped ordering。**
  A:重排 program 执行顺序,让同一时间窗内常驻 SM 的 program 集中访问一小块 A/B 数据,使工作集落进 L2。不改变计算结果与总计算量,只提高缓存命中。账面:行优先一波读 1+40 个条带,grouped 一波读 8+5 个,工作集缩小数倍。
- **Q:occupancy 是什么?它高就一定快吗?**
  A:SM 上实际常驻 warp 数与硬件上限之比,反映延迟掩盖能力。不一定——大块低 occupancy 但复用率高的配置可能更快(寄存器里的数据复用比线程并行更省带宽);最优是平衡点,所以要 autotune 而不是背口诀。